# Retrieval Evaluation — Baseline Analysis

This notebook analyzes the baseline performance of the semantic retriever.

Baseline configuration:

- Embedding model: `paraphrase-multilingual-MiniLM-L12-v2`
- Knowledge base: `context.txt`
- Retrieval representation: manually defined search keys
- Retrieval threshold: 0.30
- Evaluation dataset: 100 manually curated queries

The analysis separates:
1. ranking performance,
2. retrieval/rejection decisions,
3. performance across query categories and difficulty levels,
4. out-of-domain behavior.

## Imports and loading data

We start importing the libraries:

In [1]:
import json
from pathlib import Path

import pandas as pd

We continue loading the results:

In [2]:
# Load results
RESULTS_PATH = Path("retrieval_results.json")

with RESULTS_PATH.open("r", encoding="utf-8") as file:
    results = json.load(file)

Check:

In [3]:
print("Type: ", type(results))
print("len: ", len(results))
# Example
results[0]

Type:  <class 'list'>
len:  100


{'case_id': 'fever_001',
 'query': 'Tengo fiebre',
 'expected_id': 'fever',
 'should_retrieve': True,
 'query_type': 'direct',
 'difficulty': 'easy',
 'predicted_id': 'fever',
 'best_score': 0.6667430400848389,
 'retrieved_id': 'fever',
 'expected_rank': 1,
 'retrieval_correct': True,
 'decision_correct': True,
 'ranking': [{'rank': 1, 'id': 'fever', 'score': 0.6667430400848389},
  {'rank': 2, 'id': 'respiratory', 'score': 0.6096863746643066},
  {'rank': 3, 'id': 'headache', 'score': 0.5492120385169983},
  {'rank': 4, 'id': 'abdominal', 'score': 0.4260847568511963},
  {'rank': 5, 'id': 'greeting', 'score': 0.24636825919151306},
  {'rank': 6, 'id': 'hours_location', 'score': 0.18111678957939148},
  {'rank': 7, 'id': 'trauma', 'score': 0.1654168963432312},
  {'rank': 8, 'id': 'appointments', 'score': 0.0965028926730156},
  {'rank': 9, 'id': 'out_of_domain', 'score': 0.011328473687171936},
  {'rank': 10, 'id': 'insurance_payments', 'score': -0.027254842221736908}]}

## Building the dataframe

In [4]:
df = pd.DataFrame(results)
df.head()

,case_id,query,expected_id,should_retrieve,query_type,difficulty,predicted_id,best_score,retrieved_id,expected_rank,retrieval_correct,decision_correct,ranking
0,fever_001,Tengo fiebre,fever,True,direct,easy,fever,0.666743,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.6667430..."
1,fever_002,Tengo 38.5 de temperatura,fever,True,direct,easy,fever,0.609955,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.6099554..."
2,fever_003,El termómetro me marca 38.7 desde anoche,fever,True,paraphrase,medium,fever,0.520986,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.5209861..."
3,fever_004,Estoy con escalofríos y siento el cuerpo muy c...,fever,True,symptom_description,medium,fever,0.711315,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.7113145..."
4,fever_005,Creo que levanté temperatura y estoy bastante ...,fever,True,paraphrase,medium,fever,0.584605,fever,1.0,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.5846052..."


In [5]:
df.columns

Index(['case_id', 'query', 'expected_id', 'should_retrieve', 'query_type',
       'difficulty', 'predicted_id', 'best_score', 'retrieved_id',
       'expected_rank', 'retrieval_correct', 'decision_correct', 'ranking'],
      dtype='str')

## Sanity checks

We start with some sanity checks, in order to explore the dataframe.

In [6]:
df.shape

(100, 13)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   case_id            100 non-null    str    
 1   query              100 non-null    str    
 2   expected_id        80 non-null     str    
 3   should_retrieve    100 non-null    bool   
 4   query_type         100 non-null    str    
 5   difficulty         100 non-null    str    
 6   predicted_id       100 non-null    str    
 7   best_score         100 non-null    float64
 8   retrieved_id       79 non-null     str    
 9   expected_rank      80 non-null     float64
 10  retrieval_correct  100 non-null    bool   
 11  decision_correct   100 non-null    bool   
 12  ranking            100 non-null    object 
dtypes: bool(3), float64(2), object(1), str(7)
memory usage: 8.2+ KB


In [8]:
df["difficulty"].value_counts()

difficulty
hard      39
medium    36
easy      25
Name: count, dtype: int64

In [9]:
df["query_type"].value_counts()

query_type
paraphrase                20
direct                    18
unknown_out_of_domain     10
unsupported_medical       10
symptom_description        9
colloquial                 9
explicit_out_of_domain     8
noisy                      7
intent_description         4
indirect                   3
greeting_with_noise        2
Name: count, dtype: int64

In [10]:
df["expected_id"].value_counts(dropna=False)

expected_id
NaN                   20
fever                  8
headache               8
respiratory            8
abdominal              8
trauma                 8
appointments           8
hours_location         8
insurance_payments     8
greeting               8
out_of_domain          8
Name: count, dtype: int64

In [11]:
df.groupby(["should_retrieve", "difficulty"]).size()

should_retrieve  difficulty
False            easy           5
                 hard           9
                 medium         6
True             easy          20
                 hard          30
                 medium        30
dtype: int64

We can see some easy metrics.

In [12]:
df["retrieval_correct"].mean()

np.float64(0.63)

In [13]:
df["decision_correct"].mean()

np.float64(0.79)

Remember that `retrieval_correct` tells us whether the system returned the correct ID
(or correctly returned no ID), while `decision_correct` only tells us whether the
system made the correct retrieve-vs-reject decision, regardless of which ID was retrieved.

In [14]:
assert len(df) == 100
assert df["case_id"].is_unique
assert df["ranking"].map(len).eq(10).all()

## Ranking metrics

We first evaluate the ranking produced by the embedding model independently
of the retrieval threshold.

These metrics are computed only for queries with a known relevant document
(`expected_id != None`), since out-of-domain queries with no expected document
do not have a meaningful correct rank.

In [15]:
# Create subset with the expected id not nan
ranking_df = df[df["expected_id"].notna()].copy()
ranking_df["expected_rank"] = ranking_df["expected_rank"].astype(int)

ranking_df.shape

(80, 13)

In [16]:
ranking_df.head()

,case_id,query,expected_id,should_retrieve,query_type,difficulty,predicted_id,best_score,retrieved_id,expected_rank,retrieval_correct,decision_correct,ranking
0,fever_001,Tengo fiebre,fever,True,direct,easy,fever,0.666743,fever,1,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.6667430..."
1,fever_002,Tengo 38.5 de temperatura,fever,True,direct,easy,fever,0.609955,fever,1,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.6099554..."
2,fever_003,El termómetro me marca 38.7 desde anoche,fever,True,paraphrase,medium,fever,0.520986,fever,1,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.5209861..."
3,fever_004,Estoy con escalofríos y siento el cuerpo muy c...,fever,True,symptom_description,medium,fever,0.711315,fever,1,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.7113145..."
4,fever_005,Creo que levanté temperatura y estoy bastante ...,fever,True,paraphrase,medium,fever,0.584605,fever,1,True,True,"[{'rank': 1, 'id': 'fever', 'score': 0.5846052..."


### Top-1 Accuracy

Top-1 Accuracy measures the proportion of queries for which the correct document is ranked first.

In [17]:
ranking_df["top1_correct"] = ranking_df["expected_rank"] == 1

top1_accuracy = ranking_df["top1_correct"].mean()

top1_accuracy

np.float64(0.75)

### Recall@k

Recall@k measures whether the relevant document appears among the first $k$ retrieved results.

Since each query in this benchmark has exactly one relevant document, Recall@k is equivalent to Hit Rate@k.

In this setting, Recall@1 is equivalent to Top-1 Accuracy.

In [18]:
def recall_at_k(data, k):
    return (data["expected_rank"] <= k).mean()

In [19]:
recall_at_1 = recall_at_k(ranking_df, 1)
recall_at_2 = recall_at_k(ranking_df, 2)
recall_at_3 = recall_at_k(ranking_df, 3)
recall_at_5 = recall_at_k(ranking_df, 5)

recall_at_1, recall_at_2, recall_at_3, recall_at_5

(np.float64(0.75), np.float64(0.9), np.float64(0.925), np.float64(1.0))

### Mean Reciprocal Rank — MRR

MRR measures how high the relevant document appears in the ranking.

For each query, the reciprocal rank is:

$$
RR = \frac{1}{\text{rank}}
$$

Therefore, a relevant document ranked first contributes `1.0`, one ranked second contributes `0.5`, one ranked third contributes `0.33`, and so on.

MRR is the mean reciprocal rank across all evaluated queries.

In [20]:
ranking_df["reciprocal_rank"] = 1 / ranking_df["expected_rank"]

mrr = ranking_df["reciprocal_rank"].mean()

mrr

np.float64(0.8514583333333334)

### Saving metrics

In [21]:
ranking_metrics = pd.Series(
    {
        "Top-1 Accuracy": top1_accuracy,
        "Recall@2": recall_at_k(ranking_df, 2),
        "Recall@3": recall_at_k(ranking_df, 3),
        "Recall@5": recall_at_k(ranking_df, 5),
        "MRR": mrr,
    }
)

ranking_metrics

Top-1 Accuracy    0.750000
Recall@2          0.900000
Recall@3          0.925000
Recall@5          1.000000
MRR               0.851458
dtype: float64

### Performance by difficulty

Now we will group by the difficulty and see how the model works.

In [22]:
ranking_by_difficulty = (
    ranking_df.groupby("difficulty")["top1_correct"]
    .agg(["mean", "count"])
    .rename(
        columns={
            "mean": "top1_accuracy",
            "count": "n_cases",
        }
    )
    .reindex(["easy", "medium", "hard"])
)

ranking_by_difficulty

,top1_accuracy,n_cases
difficulty,,
easy,0.850000,20
medium,0.800000,30
hard,0.633333,30


#### Observation

Top-1 Accuracy decreases with query difficulty:

- Easy: 85.0%
- Medium: 80.0%
- Hard: 63.3%

This indicates that the manually defined difficulty levels capture a real
increase in retrieval complexity. The largest degradation occurs for queries
with less lexical overlap, indirect phrasing, colloquial language, or noise.

### Performance by category

We do the same but grouping by category.

In [23]:
ranking_by_category = (
    ranking_df.groupby("expected_id")["top1_correct"]
    .agg(["mean", "count"])
    .rename(
        columns={
            "mean": "top1_accuracy",
            "count": "n_cases",
        }
    )
    .sort_values("top1_accuracy")
)

ranking_by_category

,top1_accuracy,n_cases
expected_id,,
out_of_domain,0.500,8
trauma,0.500,8
respiratory,0.625,8
insurance_payments,0.750,8
headache,0.750,8
hours_location,0.750,8
appointments,0.875,8
fever,0.875,8
abdominal,0.875,8


#### Observation

Retrieval performance varies substantially across categories. `greeting`
achieves perfect Top-1 performance in this benchmark, while `trauma` and
`out_of_domain` obtain the lowest Top-1 Accuracy.

This suggests that the errors are not uniformly distributed across the
knowledge base. The next step is to inspect which categories are being
confused and why.